In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

In [ ]:
def load_data():
    PHI_COMMON = np.linspace(0.05, 0.55, 100)   # adjust to phi range

    # Load params table
    params_df = pd.read_csv("doe_params.csv", index_col=0)   # columns: run_id, mdot, dh_mid, ...

    runs = []
    for row in params_df.itertuples():
        # Extract data for filename
        index = row.index
        n_blades = row.n_blade
        mdot = row.mdot
        DHmid = row.DH_mid
        INC = row.incidence
        Vexp = row.Vexp
        lean_max = row.lean_compound
        lean_weighting = 1.0
        lean_straight = row.lean_straight
        tip_clearance_percent = row.tip_clearance
        filename = f"DOE_{index}_{n_blades}_{mdot}_{DHmid}_{INC}_{Vexp}_{lean_max}_{lean_weighting}_{lean_straight}_{tip_clearance_percent}"
        
        # Create interpolation from existing datapoints
        curve_df = pd.read_csv(f"data/doe_data/{filename}blade.csv")
        f = interp1d(curve_df["flow_coefficient_mean"], curve_df["pressure_rise_coefficient_mean"], kind="cubic", fill_value="extrapolate")
        
        # Save params and psi values in list
        data_as_dict = row._asdict()
        data_as_dict["psi"] = f(PHI_COMMON)
        runs.append(data_as_dict)
        
    return runs